In [1]:
# Cell 1 - Import libraries
import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap
from sklearn.cluster import DBSCAN
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

All libraries imported successfully!


In [7]:
# Cell 2 - Generate realistic accident data for Chennai
np.random.seed(42)

hotspot_centers = {
    'Anna Salai':           (13.0569, 80.2425),
    'Koyambedu':            (13.0694, 80.1948),
    'Tambaram':             (12.9249, 80.1000),
    'ECR Thiruvanmiyur':    (12.9827, 80.2707),
    'ECR Sholinganallur':   (12.9010, 80.2279),
    'ECR Neelankarai':      (12.9500, 80.2600),
    'GST Road':             (12.9716, 80.1948),
    'Poonamallee Highway':  (13.0469, 80.1167),
    'OMR Perungudi':        (12.9616, 80.2450),
    'OMR Siruseri':         (12.8700, 80.2200),
    'Adyar':                (13.0012, 80.2565),
    'Guindy':               (13.0067, 80.2206),
    'Madhavaram':           (13.1490, 80.2340),
    'Ambattur':             (13.1143, 80.1548),
    'Porur':                (13.0350, 80.1567),
    'Velachery':            (12.9815, 80.2209),
    'Perambur':             (13.1200, 80.2350),
    'Pallavaram':           (12.9675, 80.1491),
}

accident_points = []
for area, (lat, lon) in hotspot_centers.items():
    n = 80 if any(x in area for x in ['ECR', 'OMR', 'GST', 'Poonamallee', 'Guindy']) else 50
    lats = np.random.normal(lat, 0.003, n)  # tighter spread
    lons = np.random.normal(lon, 0.003, n)
    for la, lo in zip(lats, lons):
        accident_points.append({'lat': la, 'lon': lo, 'area': area})

df_accidents = pd.DataFrame(accident_points)
print(f"Total accident points: {len(df_accidents)}")

Total accident points: 1140


In [8]:
# Cell 3 - DBSCAN Clustering
coords = df_accidents[['lat', 'lon']].values
coords_rad = np.radians(coords)

db = DBSCAN(eps=0.0005, min_samples=5, algorithm='ball_tree', metric='haversine')
df_accidents['cluster'] = db.fit_predict(coords_rad)

n_clusters = len(set(df_accidents['cluster'])) - (1 if -1 in df_accidents['cluster'].values else 0)
n_noise = list(df_accidents['cluster']).count(-1)

print(f"Number of blackspot clusters found: {n_clusters}")
print(f"Noise points: {n_noise}")
print("\nCluster sizes:")
print(df_accidents[df_accidents['cluster'] != -1]['cluster'].value_counts())

Number of blackspot clusters found: 9
Noise points: 0

Cluster sizes:
cluster
3    500
4    160
5    130
6    100
0     50
1     50
2     50
7     50
8     50
Name: count, dtype: int64


In [9]:
# Cell 4 - Create Blackspot Heatmap
m = folium.Map(location=[13.0569, 80.2425], zoom_start=11)

# Add heatmap layer
heat_data = [[row['lat'], row['lon']] for _, row in df_accidents.iterrows()]
HeatMap(heat_data, radius=15, blur=10, min_opacity=0.5).add_to(m)

# Add cluster center markers
for cluster_id in df_accidents['cluster'].unique():
    if cluster_id == -1:
        continue
    cluster_points = df_accidents[df_accidents['cluster'] == cluster_id]
    center_lat = cluster_points['lat'].mean()
    center_lon = cluster_points['lon'].mean()
    size = len(cluster_points)
    area_name = cluster_points['area'].mode()[0]
    
    folium.CircleMarker(
        location=[center_lat, center_lon],
        radius=10,
        color='red',
        fill=True,
        fill_color='red',
        fill_opacity=0.7,
        popup=f"🔴 Blackspot: {area_name}\nAccidents: {size}"
    ).add_to(m)

m.save('data/blackspot_map.html')
print("Blackspot map saved! Open data/blackspot_map.html")

Blackspot map saved! Open data/blackspot_map.html
